In [ ]:
# Cell 1: Imports (Updated for SQLAlchemy)
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from urllib.parse import quote_plus
from dotenv import load_dotenv
import os
import re
from datetime import datetime, timedelta

# Cấu hình để Pandas hiển thị đầy đủ nội dung
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 1000)

print("Cell 1: Các thư viện cần thiết đã được import.")

In [ ]:
# Cell 2: Load Data from PostgreSQL (Your improved SQLAlchemy version)

load_dotenv()
DB_TABLE_NAME = 'topcv_jobs_detailed'

print(f"\nCell 2: Đang tải dữ liệu từ bảng '{DB_TABLE_NAME}'...")

df_jobs = pd.DataFrame()

try:
    # Đọc thông tin kết nối từ biến môi trường
    dbname = os.getenv('DB_NAME')
    user = os.getenv('DB_USER')
    password = quote_plus(os.getenv('DB_PASSWORD') or '')
    host = os.getenv('DB_HOST', 'localhost')
    port = os.getenv('DB_PORT', '5432')

    # Tạo SQLAlchemy engine
    engine = create_engine(f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{dbname}")

    # Đọc dữ liệu
    with engine.connect() as connection:
        query = f"SELECT * FROM {DB_TABLE_NAME} WHERE status = 'completed';"
        df_jobs = pd.read_sql(query, connection)

    if not df_jobs.empty:
        print(f"  => Đã tải thành công {len(df_jobs)} job items (status='completed').")

        # Xử lý các cột dạng array
        array_cols = ['related_job_categories', 'required_skills_tags', 'preferred_skills_tags']
        for col in array_cols:
            if col in df_jobs.columns:
                df_jobs[col] = df_jobs[col].apply(lambda x: x if x is not None and isinstance(x, list) else [])
    else:
        print("  Cảnh báo: Không có dữ liệu (status='completed') trong DB để phân tích. Hãy đảm bảo script `detail_worker.py` đang chạy.")

except Exception as e:
    print(f"  Lỗi khi tải dữ liệu từ DB: {e}")

In [ ]:
# Cell 3: Data Overview & Initial Inspection

if not df_jobs.empty:
    print("--- (1/3) Thông tin tổng quan (df_jobs.info()) ---")
    df_jobs.info()
    
    print("\n--- (2/3) 5 dòng dữ liệu đầu tiên (df_jobs.head()) ---")
    display(df_jobs.head())
    
    print("\n--- (3/3) Thống kê các giá trị rỗng (df_jobs.isnull().sum()) ---")
    # Hiển thị tất cả các dòng, sắp xếp theo số lượng null giảm dần
    with pd.option_context('display.max_rows', None):
        display(df_jobs.isnull().sum().sort_values(ascending=False))
else:
    print("Cell 3: DataFrame rỗng, không có gì để hiển thị.")

In [ ]:
# Cell 4: Data Processing Helper Functions (100% FROM YOUR ORIGINAL NOTEBOOK)
# Đây là các hàm em đã xây dựng, được tái sử dụng nguyên vẹn.

import re
import numpy as np
import pandas as pd

# --- Hàm xử lý Lương (từ Cell 11 gốc) ---
def parse_salary_string_for_list_data(salary_text):
    if pd.isna(salary_text): return np.nan, np.nan, 'triệu VNĐ', 'Không xác định'
    text_original, text_lower = str(salary_text), str(salary_text).lower()
    min_sal, max_sal, currency, sal_type = np.nan, np.nan, 'triệu VNĐ', 'Không xác định'

    if "thoả thuận" in text_lower:
        sal_type = "Thoả thuận"; return min_sal, max_sal, currency, sal_type

    if "usd" in text_lower: currency = "USD"; text_extract = re.sub(r'[^0-9.,\\-]', '', text_lower.replace("usd",""))
    elif "triệu" in text_lower: text_extract = re.sub(r'[^0-9.,\\-]', '', text_lower.replace("triệu","").replace("vnd",""))
    else: text_extract = re.sub(r'[^0-9.,\\-]', '', text_lower)
    
    nums_str = re.findall(r'\d+[\.,]?\d*', text_extract)
    nums_float = []
    for ns in nums_str:
        cleaned_ns = ns.replace('.', '').replace(',', '.') if ',' in ns and '.' not in ns[:ns.find(',')] else ns.replace(',', '')
        try: nums_float.append(float(cleaned_ns))
        except ValueError: pass
    nums = sorted(nums_float)

    if not nums: return min_sal, max_sal, currency, sal_type

    if "tới" in text_lower or "upto" in text_lower or ("đến" in text_lower and len(nums) == 1 and "từ" not in text_lower):
        sal_type = "Tối đa"; max_sal = nums[-1] if nums else np.nan
    elif "từ" in text_lower and not any(k in text_lower for k in ["đến", "tới"]) and "-" not in text_original:
        sal_type = "Tối thiểu"; min_sal = nums[0] if nums else np.nan
    elif (("-" in text_original or "đến" in text_lower) and len(nums) >= 2) or len(nums) >= 2:
        sal_type = "Khoảng"
        if len(nums) >= 2: min_sal, max_sal = nums[0], nums[-1]
        elif len(nums) == 1: min_sal = max_sal = nums[0]; sal_type = "Cố định"
        if pd.notna(min_sal) and min_sal == max_sal: sal_type = "Cố định"
    elif len(nums) == 1:
        sal_type = "Cố định"; min_sal = max_sal = nums[0]
    return min_sal, max_sal, currency, sal_type

# --- Hàm xử lý Kinh nghiệm (từ Cell 11 gốc) ---
def standardize_experience_for_list_data(exp_str):
    if pd.isna(exp_str): return 'Không xác định'
    exp_lower = str(exp_str).lower()
    if any(k in exp_lower for k in ["không yêu cầu", "chưa có kinh nghiệm", "no experience", "fresher", "mới tốt nghiệp"]): return "Không yêu cầu"
    if m := re.search(r'(?:dưới|<)\s*(\d+)\s*(?:năm|year)', exp_lower): return f"< {m.group(1)} năm"
    if m := re.search(r'(?:từ\s*)?(\d+)\s*(?:-|đến)\s*(\d+)\s*(?:năm|year)', exp_lower): return f"{m.group(1)}-{m.group(2)} năm"
    if m := re.search(r'^\d+\s*(?:năm|year)', exp_lower): return f"{m.group(0).strip()} năm" # Chỉnh 1 chút để bắt '1 năm'
    if m := re.search(r'(?:trên|>)\s*(\d+)\s*(?:năm|year)', exp_lower): return f"> {m.group(1)} năm"
    return exp_str # Trả về gốc nếu không khớp

# --- Hàm join list thành string (từ Cell 13 gốc) ---
def join_list_elements(data_list):
    if isinstance(data_list, list): return ', '.join(str(item) for item in data_list) if data_list else None
    return str(data_list) if pd.notna(data_list) else None

# --- Danh sách và hàm trích xuất Kỹ năng (từ Cell 13 gốc) ---
MASTER_SKILL_LIST = [
    "python", "sql", "java", "scala", "r", "c++", "c#", "javascript", "typescript", "php", "swift", "kotlin", "go", "ruby", "rust",
    "pandas", "numpy", "scipy", "scikit-learn", "sklearn", "tensorflow", "keras", "pytorch", "opencv", "nltk", "spacy", "gensim",
    "spark", "pyspark", "hadoop", "mapreduce", "hive", "pig", "kafka", "flink", "storm", "airflow", "luigi", "azkaban", "nifi",
    "docker", "kubernetes", "k8s", "openshift", "ansible", "terraform", "jenkins", "git", "cicd", "ci/cd",
    "aws", "azure", "gcp", "cloud", "s3", "ec2", "lambda", "rds", "dynamodb", "redshift", "glue", "emr", "sagemaker", 
    "azure data factory", "azure databricks", "azure synapse", "google bigquery", "google dataflow", "google dataproc",
    "excel", "vba", "power bi", "powerbi", "tableau", "qlik", "qlikview", "qliksense", "google data studio", "superset", "metabase", "looker",
    "ssis", "ssas", "ssrs", "informatica", "talend", "datastage",
    "etl", "elt", "data pipeline", "data modeling", "data modelling", "data warehouse", "dwh", "data lake", "data mart", "olap",
    "data analysis", "data analyst", "business intelligence", "bi", "business analyst", "ba",
    "data visualization", "data mining", "machine learning", "ml", "deep learning", "dl", "artificial intelligence", "ai", "nlp", "natural language processing",
    "statistics", "statistical modeling", "a/b testing", "hypothesis testing",
    "api", "restful api", "graphql", "microservices", "linux", "unix", "bash", "shell scripting",
    "nosql", "mongodb", "cassandra", "redis", "elasticsearch", "neo4j", "agile", "scrum", "kanban",
    "english", "tiếng anh", "giao tiếp", "communication", "presentation", "trình bày", 
    "problem solving", "critical thinking", "analytical skills", "phân tích", "tư duy logic", "teamwork", "làm việc nhóm"
] 

def extract_master_skills(row, skill_list_master):
    text_jd = str(row.get('job_description_text','')) if pd.notna(row.get('job_description_text')) else ''
    text_req = str(row.get('job_requirements_text','')) if pd.notna(row.get('job_requirements_text')) else ''
    text_combined = (text_jd + " " + text_req).lower()
    if not text_combined.strip(): return None
    found_skills = []
    for skill_keyword in skill_list_master:
        pattern = r'\b' + re.escape(skill_keyword.lower()) + r'\b'
        if re.search(pattern, text_combined):
            found_skills.append(skill_keyword) 
    return list(set(found_skills)) if found_skills else []

print("Cell 4: Các hàm helper xử lý dữ liệu gốc đã được định nghĩa.")

In [ ]:
# Cell 5: Apply Processing Functions to Create New Features

if not df_jobs.empty:
    print("--- Bắt đầu quá trình xử lý và tạo cột mới ---")
    
    # Tạo một bản sao để xử lý, giữ lại df_jobs gốc nếu cần
    df_processed = df_jobs.copy()

    # 1. Xử lý Lương (từ salary_raw_list)
    if 'salary_raw_list' in df_processed.columns:
        salary_cols = ['salary_min', 'salary_max', 'salary_currency', 'salary_type']
        df_processed[salary_cols] = df_processed['salary_raw_list'].apply(
            lambda x: pd.Series(parse_salary_string_for_list_data(x), index=salary_cols)
        )
        print("  (1/6) Đã xử lý cột 'salary_raw_list'.")

    # 2. Xử lý Kinh nghiệm (ưu tiên 'job_level' từ trang chi tiết)
    # TẠO CỘT 'experience_standardized' Ở ĐÂY
    if 'job_level' in df_processed.columns:
        df_processed['experience_standardized'] = df_processed['job_level'].apply(standardize_experience_for_list_data)
        print("  (2/6) Đã xử lý cột 'job_level' thành 'experience_standardized'.")

    # 3. Chuyển đổi Ngày tháng
    if 'application_deadline_date' in df_processed.columns:
        df_processed['deadline_dt'] = pd.to_datetime(df_processed['application_deadline_date'], format='%d/%m/%Y', errors='coerce')
        print("  (3/6) Đã xử lý 'application_deadline_date'.")

    # 4. Trích xuất Kỹ năng từ Text
    text_cols_for_skills = ['job_description_text', 'job_requirements_text']
    if all(c in df_processed.columns for c in text_cols_for_skills):
        df_processed['skills_from_text'] = df_processed.apply(extract_master_skills, args=(MASTER_SKILL_LIST,), axis=1)
        print("  (4/6) Đã trích xuất kỹ năng từ text JD.")

    # 5. Kết hợp tất cả các kỹ năng
    df_processed['skills_all'] = df_processed.apply(
        lambda row: sorted(list(set(
            row.get('required_skills_tags', []) + 
            row.get('preferred_skills_tags', []) + 
            row.get('skills_from_text', [])
        ))), axis=1
    )
    print("  (5/6) Đã kết hợp tất cả các nguồn kỹ năng vào 'skills_all'.")
    
    # 6. Tạo phiên bản chuỗi của cột skills_all để dễ hiển thị
    df_processed['skills_all_str'] = df_processed['skills_all'].apply(join_list_elements)
    print("  (6/6) Đã tạo 'skills_all_str'.")
    
    print("\n--- Xử lý hoàn tất! DataFrame 'df_processed' đã sẵn sàng để phân tích. ---")
else:
    print("Cell 5: DataFrame rỗng, bỏ qua xử lý.")

In [ ]:
# Cell 6: Phân tích các biến phân loại (Categorical Variables)

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 7)
plt.rcParams['font.sans-serif'] = ['Arial']

if 'df_processed' in locals() and not df_processed.empty:
    print("--- Phân tích các đặc tính công việc chính ---")
    
    # 1. Phân tích Yêu cầu Kinh nghiệm
    if 'experience_standardized' in df_processed.columns:
        plt.figure()
        exp_counts = df_processed['experience_standardized'].value_counts().nlargest(10)
        sns.barplot(x=exp_counts.values, y=exp_counts.index, palette='viridis')
        plt.title('Top 10 Yêu Cầu Kinh Nghiệm Phổ Biến Nhất', fontsize=16)
        plt.xlabel('Số lượng tin tuyển dụng', fontsize=12)
        plt.ylabel('Yêu cầu kinh nghiệm', fontsize=12)
        plt.show()
    else:
        print("Cảnh báo: Không tìm thấy cột 'experience_standardized' để phân tích.")

    # ... (các phần phân tích job_level, employment_type giữ nguyên như cũ)
    # 2. Phân tích Cấp bậc (Job Level)
    if 'job_level' in df_processed.columns:
        plt.figure()
        level_counts = df_processed['job_level'].value_counts().nlargest(10)
        sns.barplot(x=level_counts.values, y=level_counts.index, palette='plasma')
        plt.title('Top 10 Cấp Bậc Được Tuyển Dụng Nhiều Nhất', fontsize=16)
        plt.xlabel('Số lượng tin tuyển dụng', fontsize=12)
        plt.ylabel('Cấp bậc', fontsize=12)
        plt.show()

else:
    print("Cell 6: Không có DataFrame đã xử lý để phân tích.")

In [ ]:
# Cell 8: Phân tích Mức lương (Salary Analysis)

if 'df_processed' in locals() and not df_processed.empty and 'salary_min' in df_processed.columns:
    print("--- Phân tích phân bổ mức lương ---")
    
    # Lọc ra các jobs có thông tin lương rõ ràng (không phải 'Thoả thuận' và có giá trị)
    df_salary = df_processed[df_processed['salary_min'].notna()].copy()

    # Chuyển đổi lương USD sang VND (giả sử tỷ giá 25000)
    usd_mask = df_salary['salary_currency'] == 'USD'
    df_salary.loc[usd_mask, 'salary_min'] = df_salary.loc[usd_mask, 'salary_min'] * 25 # Lương đang tính theo triệu
    
    # Giới hạn mức lương để biểu đồ dễ nhìn hơn (loại bỏ các outlier quá lớn)
    salary_cap = 100 # triệu VND
    df_salary_plot = df_salary[df_salary['salary_min'] <= salary_cap]

    # 1. Vẽ biểu đồ Histogram phân phối mức lương tối thiểu (salary_min)
    plt.figure(figsize=(14, 7))
    sns.histplot(df_salary_plot['salary_min'], bins=30, kde=True, color='skyblue')
    
    # Thêm đường kẻ dọc cho mức lương trung bình và trung vị
    plt.axvline(df_salary_plot['salary_min'].mean(), color='red', linestyle='--', linewidth=2, label=f"Mean: {df_salary_plot['salary_min'].mean():.1f} tr")
    plt.axvline(df_salary_plot['salary_min'].median(), color='green', linestyle='-', linewidth=2, label=f"Median: {df_salary_plot['salary_min'].median():.1f} tr")
    
    plt.title(f'Phân Phối Mức Lương Tối Thiểu (đến {salary_cap} triệu)', fontsize=16)
    plt.xlabel('Mức lương tối thiểu (triệu VNĐ)', fontsize=12)
    plt.ylabel('Số lượng tin tuyển dụng', fontsize=12)
    plt.legend()
    plt.show()

    # 2. Xem xét các loại lương
    plt.figure(figsize=(10, 6))
    salary_type_counts = df_processed['salary_type'].value_counts()
    sns.barplot(x=salary_type_counts.index, y=salary_type_counts.values, palette='magma')
    plt.title('Phân Bố Các Loại Lương', fontsize=16)
    plt.xlabel('Loại lương', fontsize=12)
    plt.ylabel('Số lượng tin tuyển dụng', fontsize=12)
    plt.show()
    
else:
    print("Cell 8: Không có dữ liệu lương để phân tích.")

In [ ]:
# Cell 9: Phân tích Kỹ năng (Core Skill Analysis)

from collections import Counter

if 'df_processed' in locals() and not df_processed.empty and 'skills_all' in df_processed.columns:
    print("--- Phân tích các kỹ năng được yêu cầu nhiều nhất ---")
    
    # Dùng hàm explode để "làm phẳng" list các kỹ năng thành từng dòng
    # Sau đó bỏ các giá trị rỗng (nếu có)
    all_skills_list = df_processed['skills_all'].explode().dropna()

    if not all_skills_list.empty:
        # Đếm tần suất xuất hiện của mỗi kỹ năng
        skill_counts = Counter(all_skills_list)
        
        # Chuyển kết quả đếm thành một DataFrame để dễ dàng thao tác
        df_skill_counts = pd.DataFrame(skill_counts.items(), columns=['skill', 'count']).sort_values(by='count', ascending=False)
        
        print("\nTop 30 kỹ năng được yêu cầu nhiều nhất:")
        display(df_skill_counts.head(30))
        
        # Vẽ biểu đồ cho top 20 kỹ năng
        plt.figure(figsize=(12, 10))
        top_20_skills = df_skill_counts.head(20)
        
        # Sắp xếp lại để kỹ năng có count cao nhất ở trên cùng
        sns.barplot(x='count', y='skill', data=top_20_skills.sort_values(by='count', ascending=False), palette='coolwarm')
        
        plt.title('Top 20 Kỹ Năng "Hot" Nhất Trên Thị Trường', fontsize=16)
        plt.xlabel('Số lần xuất hiện trong tin tuyển dụng', fontsize=12)
        plt.ylabel('Kỹ năng', fontsize=12)
        plt.show()

    else:
        print("Không có dữ liệu kỹ năng để phân tích.")
else:
    print("Cell 9: Cột 'skills_all' không tồn tại hoặc DataFrame rỗng.")